In [174]:
# imports
import quail
import pliers
import numpy as np
import pandas as pd
import hypertools as hyp
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import resample
import wikipedia
import wikia

%matplotlib inline
sns.set_context('poster')
plt.rc('figure', figsize=(12, 8))

In [161]:
# load sherlock dataset, drop extraneous info
sherlock_text = pd.read_excel('../../sherlock_behavioral_data/Sherlock_Segments_1000_NN_2017.xlsx')
sherlock_text['Scene Segments'].fillna(method='ffill', inplace=True)
sherlock_text = sherlock_text[sherlock_text['Segment Number']>7]
sherlock_text = sherlock_text.reset_index(drop=True)
sherlock_text['Segment Number'] = pd.RangeIndex(len(sherlock_text.index))

## Model movie from hand annotations

In [163]:
# choose features for training model
scene_details = sherlock_text.loc[:,'Scene Details - A Level ':'Words on Screen '].apply(lambda x: ', '.join(x.fillna('')), axis=1).values.tolist()

In [164]:
# create sliding window (size 50 samples)
movie50 = []
wsize=50
for idx, sentence in enumerate(scene_details):
    movie50.append(','.join(scene_details[idx:idx+wsize]))

# vectorizer parameters
vectorizer = {
    'model' : 'CountVectorizer', 
    'params' : {
        'stop_words' : 'english'
    }
}

# topic model parameters
semantic = {
    'model' : 'LatentDirichletAllocation', 
    'params' : {
        'n_components' : 100,
        'learning_method' : 'batch',
        'random_state' : 0,
    }
}

In [165]:
# create movie model with hypertools
movie_model = hyp.tools.format_data(movie50, vectorizer=vectorizer, semantic=semantic, corpus=movie50)[0]

In [166]:
# description are by scene, not TR so stretch the model to be in TRs
ranges =[[d['Start Time (TRs, 1.5s)'],d['End Time (TRs, 1.5s)']] for i, d in sherlock_text.iterrows()]
expanded = []
for i in range(1976):
    try:
        idx = np.where([i>=r[0] and i<=r[1] for r in ranges])[0][0]
        expanded.append(movie_model[idx, :])
    except:
        expanded.append(movie_model[0, :])
movie_model = np.array(expanded)

## What features are most important to model?

In [191]:
features = sherlock_text.loc[:,'Scene Details - A Level ':'Words on Screen ']
dists = {}
for feat in features.columns:
    text_samples = features.drop(feat, axis=1).apply(lambda x: ','.join(x.fillna('')), axis=1).values.tolist()
    movie_dropone = get_movie_model(text_samples)
    dists[feat]= corr(pd.DataFrame(movie_model).T.corr().as_matrix().ravel(), pd.DataFrame(movie_dropone).T.corr().as_matrix().ravel())[0]
    print(feat)

NameError: name 'get_movie_model' is not defined

In [189]:
sns.set_palette('muted')
pd.Series(dists).sort_values().plot(kind='bar', ylim=[0, 1])
plt.ylabel('Correlation with full model')
plt.xlabel('Feature removed')

NameError: name 'dists' is not defined

## automatically model movie

In [ ]:
from pliers.extractors import GoogleVisionAPIFaceExtractor

ext = GoogleVisionAPIFaceExtractor()


In [167]:
# # similariy model subject recalls
# # loop over subjects
# recall5 = []
# wsize=10
# for sub in range(1, 18):
    
#     # load subject data
#     recall = pd.read_csv('../../sherlock_behavioral_data/NN'+str(sub)+' transcript.txt', header=None, sep='.', error_bad_lines=False, encoding='latin-1').values.tolist()[0][:-1]
    
#     rs = []  
#     # loop over sentences
#     for sentence in recall:
#         try:
#             s = sentence.encode('utf-8').strip()
#             rs.append(sentence)
#         except:
#             pass # skips over nans
    
#     # create overlapping windows of n sentences
#     sub_recall5 = []
#     for idx, sentence in enumerate(rs):
#         sub_recall5.append(','.join(rs[idx:idx+wsize]))
        
#     recall5.append(sub_recall5)
    
# # create recall models
# recall_models = hyp.tools.format_data(recall5, vectorizer=vectorizer, semantic=semantic, corpus=movie50)
    
# recall_models_rs = list(map(lambda x: resample(x, 1976), recall_models))

In [179]:
# # train model on episode's wikipedia page
# wiki_text = wikipedia.page('A_Study_in_Pink').content.split('==')[2].replace('\n','').split('.')
# wsize=5
# wiki_recall5 = []
# for idx, sentence in enumerate(wiki_text):
#     wiki_recall5.append(','.join(wiki_text[idx:idx+wsize]))
# wiki_model = hyp.tools.format_data(wiki_recall5, vectorizer=vectorizer, semantic=semantic, corpus=movie50)

In [ ]:
# #train model on episode's bakerstreet.wikia page
# wikia_text = wikia.page('bakerstreet','A_Study_in_Pink').content.replace('\n','').replace('\xa0',' ')[516:].split('.')
# wsize=12
# wikia_recall5 = []
# for idx, sentence in enumerate(wikia_text):
#     wikia_recall5.append(','.join(wikia_text[idx:idx+wsize]))
# wikia_model = hyp.tools.format_data(wikia_recall5, vectorizer=vectorizer, semantic=semantic, corpus=movie50)